In [1]:
# ============================================================
# RG3 1D CNN 시계열 불량 분류
# - RH 데이터만 사용
# - 최근 3 / 5 / 10 cycle 비교
# - 과거 Session -> 미래 Session 검증
# - 원본 센서만 CNN 입력
# - Class imbalance: pos_weight
# - Seed 3개 Ensemble
# ============================================================


# ============================================================
# 1. Library
# ============================================================

import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# ============================================================
# 2. Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# 3. Seed
# ============================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 4. 데이터 불러오기
# ============================================================

rg3 = pd.read_csv(
    r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_rg3_다시되돌림.csv"
)

print("원본 shape:", rg3.shape)


# ============================================================
# 5. 기본 전처리
# ============================================================

# TimeStamp 날짜형 변환
rg3["TimeStamp"] = pd.to_datetime(
    rg3["TimeStamp"]
)

# 시간 순 정렬
rg3 = (
    rg3
    .sort_values(
        ["TimeStamp", "original_index"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 완전 상수 컬럼 제거
# ------------------------------------------------------------

if "Clamp_Open_Position" in rg3.columns:
    rg3 = rg3.drop(
        columns=["Clamp_Open_Position"]
    )


# ------------------------------------------------------------
# PART_NAME 제거
# RH_LH와 같은 정보를 가지고 있으므로 삭제
# ------------------------------------------------------------

if "PART_NAME" in rg3.columns:
    rg3 = rg3.drop(
        columns=["PART_NAME"]
    )


# ============================================================
# 6. RH만 사용
# ============================================================

rg3_rh = (
    rg3[
        rg3["RH_LH"] == "RH"
    ]
    .copy()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)

print("\nRH shape:", rg3_rh.shape)

print("\nTarget 분포")
print(
    rg3_rh["PassOrFail"]
    .value_counts()
)

print("\nTarget 비율 (%)")
print(
    rg3_rh["PassOrFail"]
    .value_counts(normalize=True)
    * 100
)


# ============================================================
# 7. RH_LH 제거
# ============================================================

if "RH_LH" in rg3_rh.columns:
    rg3_rh = rg3_rh.drop(
        columns=["RH_LH"]
    )


# ============================================================
# 8. 생산 시간 간격 계산
# ============================================================

rg3_rh["gap_min"] = (
    rg3_rh["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)


# ============================================================
# 9. 생산 Session 정의
# 5분 초과 공백 -> 새로운 Session
# ============================================================

rg3_rh["session_id"] = (
    (rg3_rh["gap_min"] > 5)
    .cumsum()
)


# ============================================================
# 10. Session 내 생산 순번
# ============================================================

rg3_rh["cycle_in_session"] = (
    rg3_rh
    .groupby("session_id")
    .cumcount()
)


# ============================================================
# 11. Session 구조 확인
# ============================================================

session_summary = (
    rg3_rh
    .groupby("session_id")
    .agg(
        start_time=(
            "TimeStamp",
            "min"
        ),
        end_time=(
            "TimeStamp",
            "max"
        ),
        count=(
            "PassOrFail",
            "size"
        ),
        defects=(
            "PassOrFail",
            "sum"
        )
    )
)

print("\n================ Session Summary ================")
display(session_summary)


# ============================================================
# 12. CNN에 사용할 '원본 센서'만 선택
#
# 중요:
# lag / diff / roll 등 기존 파생변수는 절대 사용하지 않음
# ============================================================

exclude_cols = [
    "original_index",
    "PassOrFail",
    "gap_min",
    "session_id",
    "cycle_in_session"
]

raw_sensor_cols = [
    col
    for col in rg3_rh.select_dtypes(
        include="number"
    ).columns

    if col not in exclude_cols

    and "_lag" not in col
    and "_diff" not in col
    and "_roll" not in col
]


print("\n원본 센서 수:", len(raw_sensor_cols))

print("\n원본 센서 목록")
for col in raw_sensor_cols:
    print(col)


# ============================================================
# 13. NaN 확인
# ============================================================

nan_summary = (
    rg3_rh[raw_sensor_cols]
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

print("\n================ Sensor NaN ================")
print(nan_summary)

print(
    "\n전체 Sensor NaN:",
    rg3_rh[raw_sensor_cols]
    .isna()
    .sum()
    .sum()
)


# ============================================================
# 14. 시계열 Train / Validation Split
#
# 과거 Session -> 미래 Session
# ============================================================

time_splits = [

    (
        [0, 1, 2],
        [3]
    ),

    (
        [0, 1, 2, 3],
        [4]
    ),

    (
        [0, 1, 2, 3, 4],
        [5]
    ),

    (
        [0, 1, 2, 3, 4, 5],
        [6]
    )
]


# ============================================================
# 15. Sequence 생성 함수
#
# seq_len = 5 예시
#
# t-4
# t-3
# t-2  -> 현재 t의 PassOrFail
# t-1
# t
#
# 현재 cycle 센서 포함
# => 현재 제품 품질 판정 모델
# ============================================================

def make_sequences(
    df,
    sessions,
    features,
    seq_len
):

    X_list = []
    y_list = []
    session_list = []

    temp = (
        df[
            df["session_id"]
            .isin(sessions)
        ]
        .sort_values(
            [
                "session_id",
                "TimeStamp"
            ]
        )
        .copy()
    )

    for (
        session_id,
        group
    ) in temp.groupby(
        "session_id"
    ):

        group = (
            group
            .sort_values("TimeStamp")
            .reset_index(drop=True)
        )

        X_values = (
            group[features]
            .to_numpy(
                dtype=np.float32
            )
        )

        y_values = (
            group["PassOrFail"]
            .to_numpy(
                dtype=np.float32
            )
        )

        # session 밖의 데이터를 섞지 않음
        for i in range(
            seq_len - 1,
            len(group)
        ):

            start_idx = (
                i
                - seq_len
                + 1
            )

            end_idx = (
                i + 1
            )

            X_seq = (
                X_values[
                    start_idx:
                    end_idx
                ]
            )

            X_list.append(
                X_seq
            )

            y_list.append(
                y_values[i]
            )

            session_list.append(
                session_id
            )

    return (
        np.asarray(
            X_list,
            dtype=np.float32
        ),

        np.asarray(
            y_list,
            dtype=np.float32
        ),

        np.asarray(
            session_list
        )
    )


# ============================================================
# 16. 1D CNN Model
#
# 입력:
# [Batch, Sequence, Features]
#
# Conv1D:
# [Batch, Features, Sequence]
# ============================================================

class RG3CNN(nn.Module):

    def __init__(
        self,
        n_features
    ):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv1d(
                in_channels=n_features,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.Dropout(
                p=0.2
            ),

            nn.Conv1d(
                in_channels=32,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.AdaptiveAvgPool1d(
                output_size=1
            )
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Dropout(
                p=0.2
            ),

            nn.Linear(
                in_features=16,
                out_features=1
            )
        )


    def forward(
        self,
        x
    ):

        # [B, Seq, Features]
        #
        # ->
        #
        # [B, Features, Seq]

        x = x.permute(
            0,
            2,
            1
        )

        x = self.conv(x)

        x = self.classifier(x)

        return x.squeeze(1)


# ============================================================
# 17. CNN 학습 함수
# ============================================================

def train_cnn(
    X_train,
    y_train,
    X_valid,
    seed=42,
    epochs=50,
    batch_size=32,
    lr=0.001
):

    set_seed(seed)


    # --------------------------------------------------------
    # TensorDataset
    # --------------------------------------------------------

    train_dataset = TensorDataset(

        torch.tensor(
            X_train,
            dtype=torch.float32
        ),

        torch.tensor(
            y_train,
            dtype=torch.float32
        )
    )


    # --------------------------------------------------------
    # DataLoader
    # --------------------------------------------------------

    train_loader = DataLoader(

        train_dataset,

        batch_size=batch_size,

        shuffle=True
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = RG3CNN(
        n_features=
            X_train.shape[2]
    ).to(device)


    # --------------------------------------------------------
    # Class imbalance 보정
    #
    # 정상 / 불량 비율만큼 불량 Loss 가중치 증가
    # --------------------------------------------------------

    n_pos = float(
        y_train.sum()
    )

    n_neg = float(
        len(y_train)
        - n_pos
    )

    if n_pos == 0:
        raise ValueError(
            "Train 데이터에 불량 데이터가 없습니다."
        )

    pos_weight = torch.tensor(
        [n_neg / n_pos],
        dtype=torch.float32,
        device=device
    )


    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    criterion = (
        nn.BCEWithLogitsLoss(
            pos_weight=
                pos_weight
        )
    )


    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = (
        torch.optim.AdamW(

            model.parameters(),

            lr=lr,

            weight_decay=1e-4
        )
    )


    # ========================================================
    # Training
    # ========================================================

    model.train()

    for epoch in range(
        epochs
    ):

        epoch_loss = 0

        for (
            X_batch,
            y_batch
        ) in train_loader:

            X_batch = (
                X_batch
                .to(device)
            )

            y_batch = (
                y_batch
                .to(device)
            )


            optimizer.zero_grad()


            logits = model(
                X_batch
            )


            loss = criterion(
                logits,
                y_batch
            )


            loss.backward()


            optimizer.step()


            epoch_loss += (
                loss.item()
                * len(X_batch)
            )


    # ========================================================
    # Validation Prediction
    # ========================================================

    model.eval()

    with torch.no_grad():

        X_valid_tensor = (
            torch.tensor(
                X_valid,
                dtype=torch.float32,
                device=device
            )
        )

        logits = model(
            X_valid_tensor
        )

        probs = (
            torch.sigmoid(
                logits
            )
            .cpu()
            .numpy()
        )


    return probs


# ============================================================
# 18. 실험 설정
# ============================================================

sequence_lengths = [
    3,
    5,
    10
]

# 딥러닝은 초기값에 따라 결과가 흔들릴 수 있으므로
# seed 3개 ensemble
seeds = [
    42,
    43,
    44
]


# ============================================================
# 19. CNN 시간순 Cross Validation
# ============================================================

cnn_results = []


for seq_len in sequence_lengths:

    print(
        "\n",
        "=" * 60
    )

    print(
        f"Sequence Length = {seq_len}"
    )

    print(
        "=" * 60
    )


    for fold, (
        train_sessions,
        valid_sessions
    ) in enumerate(
        time_splits,
        start=1
    ):

        # ====================================================
        # Train 기준 Scaling
        # ====================================================

        train_rows = (
            rg3_rh[
                rg3_rh[
                    "session_id"
                ].isin(
                    train_sessions
                )
            ]
            .copy()
        )


        scaler = (
            StandardScaler()
        )


        scaler.fit(
            train_rows[
                raw_sensor_cols
            ]
        )


        # ----------------------------------------------------
        # 전체 데이터 복사
        # Scaling 기준은 Train
        # ----------------------------------------------------

        temp_df = (
            rg3_rh.copy()
        )


        temp_df[
            raw_sensor_cols
        ] = scaler.transform(

            temp_df[
                raw_sensor_cols
            ]
        )


        # ====================================================
        # Train sequence
        # ====================================================

        (
            X_train,
            y_train,
            train_session_ids
        ) = make_sequences(

            temp_df,

            train_sessions,

            raw_sensor_cols,

            seq_len
        )


        # ====================================================
        # Validation sequence
        # ====================================================

        (
            X_valid,
            y_valid,
            valid_session_ids
        ) = make_sequences(

            temp_df,

            valid_sessions,

            raw_sensor_cols,

            seq_len
        )


        # ====================================================
        # NaN / Inf 확인
        # ====================================================

        print(
            f"\nSeq {seq_len}"
            f" | Fold {fold}"
        )

        print(
            "Train Shape:",
            X_train.shape
        )

        print(
            "Valid Shape:",
            X_valid.shape
        )

        print(
            "Train Defects:",
            int(
                y_train.sum()
            )
        )

        print(
            "Valid Defects:",
            int(
                y_valid.sum()
            )
        )

        print(
            "Train NaN:",
            np.isnan(
                X_train
            ).sum()
        )

        print(
            "Valid NaN:",
            np.isnan(
                X_valid
            ).sum()
        )

        print(
            "Train Inf:",
            np.isinf(
                X_train
            ).sum()
        )

        print(
            "Valid Inf:",
            np.isinf(
                X_valid
            ).sum()
        )


        # ----------------------------------------------------
        # NaN 있으면 즉시 중단
        # ----------------------------------------------------

        if (
            np.isnan(X_train).any()
            or
            np.isnan(X_valid).any()
        ):

            raise ValueError(
                "CNN 입력에 NaN이 있습니다."
            )


        if (
            np.isinf(X_train).any()
            or
            np.isinf(X_valid).any()
        ):

            raise ValueError(
                "CNN 입력에 Inf가 있습니다."
            )


        # ====================================================
        # Seed Ensemble
        # ====================================================

        predictions = []


        for seed in seeds:

            pred = train_cnn(

                X_train=
                    X_train,

                y_train=
                    y_train,

                X_valid=
                    X_valid,

                seed=
                    seed,

                epochs=
                    50,

                batch_size=
                    32,

                lr=
                    0.001
            )


            predictions.append(
                pred
            )


        # ====================================================
        # Ensemble 평균
        # ====================================================

        pred_mean = np.mean(
            predictions,
            axis=0
        )


        print(
            "Prediction NaN:",
            np.isnan(
                pred_mean
            ).sum()
        )


        # ====================================================
        # Metrics
        # ====================================================

        baseline_pr = float(
            y_valid.mean()
        )


        pr_auc = (
            average_precision_score(
                y_valid,
                pred_mean
            )
        )


        roc_auc = (
            roc_auc_score(
                y_valid,
                pred_mean
            )
        )


        cnn_results.append({

            "Seq_Len":
                seq_len,

            "Fold":
                fold,

            "Train_Sessions":
                str(
                    train_sessions
                ),

            "Valid_Session":
                valid_sessions[0],

            "N_Train":
                len(
                    y_train
                ),

            "Train_Defects":
                int(
                    y_train.sum()
                ),

            "N_Valid":
                len(
                    y_valid
                ),

            "Valid_Defects":
                int(
                    y_valid.sum()
                ),

            "Baseline_PR":
                baseline_pr,

            "PR_AUC":
                pr_auc,

            "PR_Uplift":
                (
                    pr_auc
                    - baseline_pr
                ),

            "ROC_AUC":
                roc_auc
        })


# ============================================================
# 20. Fold별 결과
# ============================================================

cnn_result_df = (
    pd.DataFrame(
        cnn_results
    )
)


print(
    "\n\n================ CNN Fold Results ================"
)

display(
    cnn_result_df
)


# ============================================================
# 21. Sequence 길이별 요약
# ============================================================

cnn_summary = (

    cnn_result_df

    .groupby(
        "Seq_Len"
    )

    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_Min=(
            "PR_AUC",
            "min"
        ),

        PR_Max=(
            "PR_AUC",
            "max"
        ),

        Uplift_Mean=(
            "PR_Uplift",
            "mean"
        ),

        Uplift_Min=(
            "PR_Uplift",
            "min"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_Min=(
            "ROC_AUC",
            "min"
        ),

        ROC_Max=(
            "ROC_AUC",
            "max"
        )
    )

    .reset_index()

    .sort_values(
        "PR_Mean",
        ascending=False
    )
)


print(
    "\n\n================ CNN Summary ================"
)

display(
    cnn_summary
)


# ============================================================
# 22. 현재 Random Forest 결과와 비교
# ============================================================

rf_pr_mean = (
    0.10809677538421754
)

rf_roc_mean = (
    0.4687362882215823
)


comparison = (
    cnn_summary[
        [
            "Seq_Len",
            "PR_Mean",
            "PR_Min",
            "ROC_Mean"
        ]
    ]
    .copy()
)


comparison[
    "RF_PR_Mean"
] = rf_pr_mean


comparison[
    "RF_ROC_Mean"
] = rf_roc_mean


comparison[
    "PR_vs_RF"
] = (
    comparison[
        "PR_Mean"
    ]
    - rf_pr_mean
)


comparison[
    "ROC_vs_RF"
] = (
    comparison[
        "ROC_Mean"
    ]
    - rf_roc_mean
)


print(
    "\n\n================ CNN vs RF ================"
)

display(
    comparison
)

Device: cpu
원본 shape: (1182, 29)

RH shape: (591, 27)

Target 분포
PassOrFail
0    566
1     25
Name: count, dtype: int64

Target 비율 (%)
PassOrFail
0    95.769882
1     4.230118
Name: proportion, dtype: float64

================ Session Summary ================


,start_time,end_time,count,defects
session_id,,,,
0,2020-10-21 00:16:26,2020-10-21 00:57:37,41,0
1,2020-10-22 00:24:05,2020-10-22 01:28:56,64,4
2,2020-10-22 01:40:17,2020-10-22 02:20:26,39,1
3,2020-10-22 04:40:26,2020-10-22 08:09:24,197,10
4,2020-10-23 01:01:28,2020-10-23 01:29:17,28,2
5,2020-10-23 01:47:49,2020-10-23 02:36:11,46,2
6,2020-10-23 04:41:48,2020-10-23 07:46:03,176,6



원본 센서 수: 23

원본 센서 목록
Injection_Time
Filling_Time
Plasticizing_Time
Cycle_Time
Clamp_Close_Time
Cushion_Position
Plasticizing_Position
Max_Injection_Speed
Max_Screw_RPM
Average_Screw_RPM
Max_Injection_Pressure
Max_Switch_Over_Pressure
Max_Back_Pressure
Average_Back_Pressure
Barrel_Temperature_1
Barrel_Temperature_2
Barrel_Temperature_3
Barrel_Temperature_4
Barrel_Temperature_5
Barrel_Temperature_6
Hopper_Temperature
Mold_Temperature_3
Mold_Temperature_4

================ Sensor NaN ================
Injection_Time              0
Filling_Time                0
Plasticizing_Time           0
Cycle_Time                  0
Clamp_Close_Time            0
Cushion_Position            0
Plasticizing_Position       0
Max_Injection_Speed         0
Max_Screw_RPM               0
Average_Screw_RPM           0
Max_Injection_Pressure      0
Max_Switch_Over_Pressure    0
Max_Back_Pressure           0
Average_Back_Pressure       0
Barrel_Temperature_1        0
Barrel_Temperature_2        0
Barrel_Temperat

,Seq_Len,Fold,Train_Sessions,Valid_Session,N_Train,Train_Defects,N_Valid,Valid_Defects,Baseline_PR,PR_AUC,PR_Uplift,ROC_AUC
0,3,1,"[0, 1, 2]",3,138,5,195,10,0.051282,0.069606,0.018324,0.574054
1,3,2,"[0, 1, 2, 3]",4,333,15,26,2,0.076923,0.075397,-0.001526,0.250000
2,3,3,"[0, 1, 2, 3, 4]",5,359,17,44,2,0.045455,0.041958,-0.003497,0.202381
3,3,4,"[0, 1, 2, 3, 4, 5]",6,403,19,174,6,0.034483,0.041975,0.007493,0.503968
4,5,1,"[0, 1, 2]",3,132,5,193,10,0.051813,0.088860,0.037046,0.593443
5,5,2,"[0, 1, 2, 3]",4,325,15,24,1,0.041667,0.062500,0.020833,0.347826
6,5,3,"[0, 1, 2, 3, 4]",5,349,16,42,2,0.047619,0.058333,0.010714,0.350000
7,5,4,"[0, 1, 2, 3, 4, 5]",6,391,18,172,6,0.034884,0.041575,0.006692,0.471888
8,10,1,"[0, 1, 2]",3,117,5,188,8,0.042553,0.083980,0.041427,0.654861
9,10,2,"[0, 1, 2, 3]",4,305,13,19,1,0.052632,0.055556,0.002924,0.055556




================ CNN Summary ================


,Seq_Len,PR_Mean,PR_Min,PR_Max,Uplift_Mean,Uplift_Min,ROC_Mean,ROC_Min,ROC_Max
2,10,0.077301,0.040257,0.129412,0.031009,0.002924,0.404882,0.055556,0.654861
1,5,0.062817,0.041575,0.088860,0.018821,0.006692,0.440789,0.347826,0.593443
0,3,0.057234,0.041958,0.075397,0.005199,-0.003497,0.382601,0.202381,0.574054




================ CNN vs RF ================


,Seq_Len,PR_Mean,PR_Min,ROC_Mean,RF_PR_Mean,RF_ROC_Mean,PR_vs_RF,ROC_vs_RF
2,10,0.077301,0.040257,0.404882,0.108097,0.468736,-0.030796,-0.063855
1,5,0.062817,0.041575,0.440789,0.108097,0.468736,-0.045280,-0.027947
0,3,0.057234,0.041958,0.382601,0.108097,0.468736,-0.050863,-0.086135
